In [0]:
path  = "/Volumes/workspace/default/ecommerece/OnlineRetail.csv"

In [0]:
       
df = spark.read.csv(path, header=True, inferSchema=True)


df.show(10)

+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|   InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|12/1/2010 8:26|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|12/1/2010 8:26|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|    22752|SET 7 BABUSHKA NE...|       2|12/1/2010 8:26|     7.65|     17850|United Kingdom|
|   536365|    21730|GLASS STAR FROSTE...|       6|12/1/2010 8:26|     4.

In [0]:
# Step 1: Clean the data
from pyspark.sql.functions import col, to_timestamp, year, month, quarter, sum, count, avg, max, min, round, desc, when
from pyspark.sql.types import DoubleType

# Clean the data
orders_cleaned = df \
    .filter(col("Quantity") > 0) \
    .filter(col("UnitPrice") > 0) \
    .filter(col("CustomerID").isNotNull()) \
    .filter(col("Description").isNotNull()) \
    .withColumn("InvoiceDate", to_timestamp(col("InvoiceDate"), "M/d/yyyy H:mm")) \
    .withColumn("TotalPrice", round(col("Quantity") * col("UnitPrice"), 2)) \
    .withColumn("Year", year(col("InvoiceDate"))) \
    .withColumn("Month", month(col("InvoiceDate"))) \
    .withColumn("Quarter", quarter(col("InvoiceDate")))

print(f"Original records: {df.count()}")
print(f"Cleaned records: {orders_cleaned.count()}")
display(orders_cleaned.limit(10))

Original records: 541909
Cleaned records: 397884


InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice,Year,Month,Quarter
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01T08:26:00.000Z,2.55,17850,United Kingdom,15.3,2010,12,4
536365,71053,WHITE METAL LANTERN,6,2010-12-01T08:26:00.000Z,3.39,17850,United Kingdom,20.34,2010,12,4
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01T08:26:00.000Z,2.75,17850,United Kingdom,22.0,2010,12,4
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01T08:26:00.000Z,3.39,17850,United Kingdom,20.34,2010,12,4
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01T08:26:00.000Z,3.39,17850,United Kingdom,20.34,2010,12,4
536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01T08:26:00.000Z,7.65,17850,United Kingdom,15.3,2010,12,4
536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01T08:26:00.000Z,4.25,17850,United Kingdom,25.5,2010,12,4
536366,22633,HAND WARMER UNION JACK,6,2010-12-01T08:28:00.000Z,1.85,17850,United Kingdom,11.1,2010,12,4
536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01T08:28:00.000Z,1.85,17850,United Kingdom,11.1,2010,12,4
536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01T08:34:00.000Z,1.69,13047,United Kingdom,54.08,2010,12,4


In [0]:
# Step 2: Monthly Sales by Country
monthly_sales_by_country = orders_cleaned \
    .groupBy("Year", "Month", "Country") \
    .agg(
        round(sum("TotalPrice"), 2).alias("total_revenue"),
        count("InvoiceNo").alias("order_count"),
        round(avg("TotalPrice"), 2).alias("avg_order_value")
    ) \
    .orderBy("Year", "Month", desc("total_revenue"))

print(f"Monthly sales records: {monthly_sales_by_country.count()}")
display(monthly_sales_by_country.limit(10))

Monthly sales records: 287


Year,Month,Country,total_revenue,order_count,avg_order_value
2010,12,United Kingdom,498661.85,23942,20.83
2010,12,Germany,15241.14,512,29.77
2010,12,France,9616.31,434,22.16
2010,12,EIRE,8813.88,333,26.47
2010,12,Netherlands,8784.48,72,122.01
2010,12,Japan,7705.07,65,118.54
2010,12,Sweden,3834.3,26,147.47
2010,12,Norway,3787.12,147,25.76
2010,12,Portugal,2439.97,116,21.03
2010,12,Spain,1843.73,75,24.58


In [0]:
# Step 3: Quarterly Sales by Country
quarterly_sales_by_country = orders_cleaned \
    .groupBy("Year", "Quarter", "Country") \
    .agg(
        round(sum("TotalPrice"), 2).alias("total_revenue"),
        count("InvoiceNo").alias("order_count"),
        round(avg("TotalPrice"), 2).alias("avg_order_value"),
        sum("Quantity").alias("total_quantity")
    ) \
    .orderBy("Year", "Quarter", desc("total_revenue"))

print(f"Quarterly sales records: {quarterly_sales_by_country.count()}")
display(quarterly_sales_by_country.limit(10))

Quarterly sales records: 138


Year,Quarter,Country,total_revenue,order_count,avg_order_value,total_quantity
2010,4,United Kingdom,498661.85,23942,20.83,267767
2010,4,Germany,15241.14,512,29.77,6877
2010,4,France,9616.31,434,22.16,4989
2010,4,EIRE,8813.88,333,26.47,5243
2010,4,Netherlands,8784.48,72,122.01,6811
2010,4,Japan,7705.07,65,118.54,4093
2010,4,Sweden,3834.3,26,147.47,3954
2010,4,Norway,3787.12,147,25.76,3582
2010,4,Portugal,2439.97,116,21.03,984
2010,4,Spain,1843.73,75,24.58,867


In [0]:
# Step 4: Overall Country-wise Sales
country_sales = orders_cleaned \
    .groupBy("Country") \
    .agg(
        round(sum("TotalPrice"), 2).alias("total_revenue"),
        count("InvoiceNo").alias("total_orders"),
        count(col("CustomerID")).alias("unique_customers"),
        sum("Quantity").alias("total_quantity_sold"),
        round(avg("TotalPrice"), 2).alias("avg_order_value"),
        round(min("TotalPrice"), 2).alias("min_order_value"),
        round(max("TotalPrice"), 2).alias("max_order_value")
    ) \
    .orderBy(desc("total_revenue"))

print(f"Country sales records: {country_sales.count()}")
display(country_sales)

Country sales records: 37


Country,total_revenue,total_orders,unique_customers,total_quantity_sold,avg_order_value,min_order_value,max_order_value
United Kingdom,7308391.55,354321,354321,4256740,20.63,0.0,168469.6
Netherlands,285446.34,2359,2359,200361,121.0,0.39,4992.0
EIRE,265545.9,7236,7236,140275,36.7,1.25,2365.2
Germany,228867.14,9040,9040,119261,25.32,0.39,876.0
France,209024.05,8341,8341,111471,25.06,0.29,4161.06
Australia,138521.31,1182,1182,83901,117.19,0.42,1718.4
Spain,61577.11,2484,2484,27940,24.79,0.21,1350.0
Switzerland,56443.95,1841,1841,30082,30.66,2.0,360.0
Belgium,41196.34,2031,2031,23237,20.28,2.5,165.0
Sweden,38378.33,451,451,36083,85.1,5.04,1188.0


In [0]:
# Step 5: Top Products Analysis
top_products = orders_cleaned \
    .groupBy("StockCode", "Description") \
    .agg(
        sum("Quantity").alias("total_quantity_sold"),
        round(sum("TotalPrice"), 2).alias("total_revenue"),
        count("InvoiceNo").alias("times_ordered"),
        round(avg("UnitPrice"), 2).alias("avg_unit_price")
    ) \
    .orderBy(desc("total_revenue"))

print(f"Product records: {top_products.count()}")
display(top_products.limit(20))

Product records: 3897


StockCode,Description,total_quantity_sold,total_revenue,times_ordered,avg_unit_price
23843,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.6,1,2.08
22423,REGENCY CAKESTAND 3 TIER,12402,142592.95,1723,12.48
85123A,WHITE HANGING HEART T-LIGHT HOLDER,36725,100448.15,2028,2.89
85099B,JUMBO BAG RED RETROSPOT,46181,85220.78,1618,2.02
23166,MEDIUM CERAMIC TOP STORAGE JAR,77916,81416.73,198,1.22
POST,POSTAGE,3120,77803.96,1099,31.57
47566,PARTY BUNTING,15291,68844.33,1396,4.88
84879,ASSORTED COLOUR BIRD ORNAMENT,35362,56580.34,1408,1.68
M,Manual,7173,53779.93,284,175.29
23084,RABBIT NIGHT LIGHT,27202,51346.2,842,2.01


In [0]:
# Step 6: Customer Summary
customer_summary = orders_cleaned \
    .groupBy("CustomerID") \
    .agg(
        count("InvoiceNo").alias("total_orders"),
        round(sum("TotalPrice"), 2).alias("total_spent"),
        round(avg("TotalPrice"), 2).alias("avg_order_value"),
        sum("Quantity").alias("total_items_purchased"),
        max("InvoiceDate").alias("last_order_date"),
        min("InvoiceDate").alias("first_order_date")
    ) \
    .orderBy(desc("total_spent"))

print(f"Customer records: {customer_summary.count()}")
display(customer_summary.limit(20))

Customer records: 4338


CustomerID,total_orders,total_spent,avg_order_value,total_items_purchased,last_order_date,first_order_date
14646,2076,280206.02,134.97,196915,2011-12-08T12:12:00.000Z,2010-12-20T10:09:00.000Z
18102,431,259657.3,602.45,64124,2011-12-09T11:50:00.000Z,2010-12-07T16:42:00.000Z
17450,337,194550.79,577.3,69993,2011-12-01T13:29:00.000Z,2010-12-07T09:23:00.000Z
16446,3,168472.5,56157.5,80997,2011-12-09T09:15:00.000Z,2011-05-18T09:52:00.000Z
14911,5675,143825.06,25.34,80265,2011-12-08T15:54:00.000Z,2010-12-01T14:05:00.000Z
12415,714,124914.53,174.95,77374,2011-11-15T14:22:00.000Z,2011-01-06T11:12:00.000Z
14156,1400,117379.63,83.84,57885,2011-11-30T10:54:00.000Z,2010-12-03T11:48:00.000Z
17511,963,91062.38,94.56,64549,2011-12-07T10:12:00.000Z,2010-12-01T10:19:00.000Z
16029,242,81024.84,334.81,40208,2011-11-01T10:27:00.000Z,2010-12-01T09:57:00.000Z
12346,1,77183.6,77183.6,74215,2011-01-18T10:01:00.000Z,2011-01-18T10:01:00.000Z


In [0]:
# Step 7: Save all tables to Unity Catalog
catalog = "workspace"
schema = "default"

# Save orders_cleaned
orders_cleaned.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.orders_cleaned")
print(f"✓ Saved {catalog}.{schema}.orders_cleaned")

# Save monthly_sales_by_country
monthly_sales_by_country.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.monthly_sales_by_country")
print(f"✓ Saved {catalog}.{schema}.monthly_sales_by_country")

# Save quarterly_sales_by_country
quarterly_sales_by_country.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.quarterly_sales_by_country")
print(f"✓ Saved {catalog}.{schema}.quarterly_sales_by_country")

# Save country_sales
country_sales.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.country_sales")
print(f"✓ Saved {catalog}.{schema}.country_sales")

# Save top_products
top_products.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.top_products")
print(f"✓ Saved {catalog}.{schema}.top_products")

# Save customer_summary
customer_summary.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.customer_summary")
print(f"✓ Saved {catalog}.{schema}.customer_summary")

print("\n🎉 All tables created successfully!")

✓ Saved workspace.default.orders_cleaned
✓ Saved workspace.default.monthly_sales_by_country
✓ Saved workspace.default.quarterly_sales_by_country
✓ Saved workspace.default.country_sales
✓ Saved workspace.default.top_products
✓ Saved workspace.default.customer_summary

🎉 All tables created successfully!


In [0]:
%sql
-- Revenue Trend Analysis: Monthly revenue progression
SELECT 
  Year,
  Month,
  SUM(total_revenue) as monthly_revenue,
  SUM(order_count) as total_orders,
  ROUND(SUM(total_revenue) / SUM(order_count), 2) as avg_order_value
FROM workspace.default.monthly_sales_by_country
GROUP BY Year, Month
ORDER BY Year, Month

Year,Month,monthly_revenue,total_orders,avg_order_value
2010,12,572713.89,26157,21.9
2011,1,569445.0399999997,21229,26.82
2011,2,447137.3499999999,19927,22.44
2011,3,595500.7600000001,27175,21.91
2011,4,469200.36,22642,20.72
2011,5,678594.5599999998,28320,23.96
2011,6,661213.69,27185,24.32
2011,7,600091.0099999998,26825,22.37
2011,8,645343.9000000003,27007,23.9
2011,9,952838.3799999999,40028,23.8


In [0]:
%sql
-- Top 10 Countries by Revenue with Market Share
SELECT 
  Country,
  total_revenue,
  total_orders,
  unique_customers,
  avg_order_value,
  ROUND(total_revenue / SUM(total_revenue) OVER () * 100, 2) as market_share_percent
FROM workspace.default.country_sales
ORDER BY total_revenue DESC
LIMIT 10

Country,total_revenue,total_orders,unique_customers,avg_order_value,market_share_percent
United Kingdom,7308391.55,354321,354321,20.63,82.01
Netherlands,285446.34,2359,2359,121.0,3.2
EIRE,265545.9,7236,7236,36.7,2.98
Germany,228867.14,9040,9040,25.32,2.57
France,209024.05,8341,8341,25.06,2.35
Australia,138521.31,1182,1182,117.19,1.55
Spain,61577.11,2484,2484,24.79,0.69
Switzerland,56443.95,1841,1841,30.66,0.63
Belgium,41196.34,2031,2031,20.28,0.46
Sweden,38378.33,451,451,85.1,0.43


In [0]:
%sql
-- Top 20 Best-Selling Products by Revenue
SELECT 
  StockCode,
  Description,
  total_quantity_sold,
  total_revenue,
  times_ordered,
  avg_unit_price,
  ROUND(total_revenue / times_ordered, 2) as revenue_per_order
FROM workspace.default.top_products
ORDER BY total_revenue DESC
LIMIT 20

StockCode,Description,total_quantity_sold,total_revenue,times_ordered,avg_unit_price,revenue_per_order
23843,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.6,1,2.08,168469.6
22423,REGENCY CAKESTAND 3 TIER,12402,142592.95,1723,12.48,82.76
85123A,WHITE HANGING HEART T-LIGHT HOLDER,36725,100448.15,2028,2.89,49.53
85099B,JUMBO BAG RED RETROSPOT,46181,85220.78,1618,2.02,52.67
23166,MEDIUM CERAMIC TOP STORAGE JAR,77916,81416.73,198,1.22,411.2
POST,POSTAGE,3120,77803.96,1099,31.57,70.8
47566,PARTY BUNTING,15291,68844.33,1396,4.88,49.32
84879,ASSORTED COLOUR BIRD ORNAMENT,35362,56580.34,1408,1.68,40.18
M,Manual,7173,53779.93,284,175.29,189.37
23084,RABBIT NIGHT LIGHT,27202,51346.2,842,2.01,60.98


In [0]:
%sql
-- Customer Segmentation: High, Medium, Low Value
SELECT 
  CASE 
    WHEN total_spent >= 10000 THEN 'High Value'
    WHEN total_spent >= 1000 THEN 'Medium Value'
    ELSE 'Low Value'
  END as customer_segment,
  COUNT(CustomerID) as customer_count,
  ROUND(SUM(total_spent), 2) as segment_revenue,
  ROUND(AVG(total_spent), 2) as avg_customer_value,
  ROUND(AVG(total_orders), 2) as avg_orders_per_customer
FROM workspace.default.customer_summary
GROUP BY 
  CASE 
    WHEN total_spent >= 10000 THEN 'High Value'
    WHEN total_spent >= 1000 THEN 'Medium Value'
    ELSE 'Low Value'
  END
ORDER BY segment_revenue DESC

customer_segment,customer_count,segment_revenue,avg_customer_value,avg_orders_per_customer
Medium Value,1564,4150126.69,2653.53,154.36
High Value,104,3653544.48,35130.24,640.84
Low Value,2670,1107736.73,414.88,33.64


In [0]:
%sql
-- Quarterly Performance Comparison
SELECT 
  Year,
  Quarter,
  SUM(total_revenue) as quarterly_revenue,
  SUM(order_count) as total_orders,
  COUNT(DISTINCT Country) as countries_served,
  ROUND(SUM(total_revenue) / SUM(order_count), 2) as avg_order_value
FROM workspace.default.quarterly_sales_by_country
GROUP BY Year, Quarter
ORDER BY Year, Quarter

Year,Quarter,quarterly_revenue,total_orders,countries_served,avg_order_value
2010,4,572713.89,26157,22,21.9
2011,1,1612083.15,68331,29,23.59
2011,2,1809008.6100000003,78147,30,23.15
2011,3,2198273.29,93860,29,23.42
2011,4,2719328.9600000004,131389,28,20.7


In [0]:
%sql
-- Customer Loyalty Analysis: One-time vs Repeat Customers
SELECT 
  CASE 
    WHEN total_orders = 1 THEN 'One-time Buyer'
    WHEN total_orders BETWEEN 2 AND 5 THEN 'Occasional (2-5 orders)'
    WHEN total_orders BETWEEN 6 AND 20 THEN 'Regular (6-20 orders)'
    ELSE 'Loyal (20+ orders)'
  END as customer_type,
  COUNT(CustomerID) as customer_count,
  ROUND(SUM(total_spent), 2) as total_revenue,
  ROUND(AVG(total_spent), 2) as avg_lifetime_value,
  ROUND(SUM(total_spent) / SUM(SUM(total_spent)) OVER () * 100, 2) as revenue_contribution_percent
FROM workspace.default.customer_summary
GROUP BY 
  CASE 
    WHEN total_orders = 1 THEN 'One-time Buyer'
    WHEN total_orders BETWEEN 2 AND 5 THEN 'Occasional (2-5 orders)'
    WHEN total_orders BETWEEN 6 AND 20 THEN 'Regular (6-20 orders)'
    ELSE 'Loyal (20+ orders)'
  END
ORDER BY total_revenue DESC

customer_type,customer_count,total_revenue,avg_lifetime_value,revenue_contribution_percent
Loyal (20+ orders),3053,8086469.0,2648.7,90.74
Regular (6-20 orders),986,439076.62,445.31,4.93
Occasional (2-5 orders),228,287905.02,1262.74,3.23
One-time Buyer,71,97957.26,1379.68,1.1


In [0]:
%sql
-- UK vs International Revenue Comparison by Month
SELECT 
  Year,
  Month,
  SUM(CASE WHEN Country = 'United Kingdom' THEN total_revenue ELSE 0 END) as uk_revenue,
  SUM(CASE WHEN Country != 'United Kingdom' THEN total_revenue ELSE 0 END) as international_revenue,
  ROUND(SUM(CASE WHEN Country != 'United Kingdom' THEN total_revenue ELSE 0 END) / 
        NULLIF(SUM(total_revenue), 0) * 100, 2) as international_percentage
FROM workspace.default.monthly_sales_by_country
GROUP BY Year, Month
ORDER BY Year, Month

Year,Month,uk_revenue,international_revenue,international_percentage
2010,12,498661.85,74052.04000000002,12.93
2011,1,442190.06,127254.98000000001,22.35
2011,2,355655.63,91481.72000000003,20.46
2011,3,467198.59,128302.17,21.55
2011,4,409559.14,59641.219999999994,12.71
2011,5,551568.82,127025.73999999998,18.72
2011,6,524915.48,136298.20999999996,20.61
2011,7,485612.25,114478.76000000004,19.08
2011,8,498453.32,146890.57999999996,22.76
2011,9,796780.27,156058.11000000002,16.38


In [0]:
# Create dedicated catalog schema for e-commerce analytics
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.ecommerce_analytics")
print("✓ Created schema: workspace.ecommerce_analytics")

✓ Created schema: workspace.ecommerce_analytics


In [0]:
# Save all base tables to ecommerce_analytics schema
catalog = "workspace"
schema = "ecommerce_analytics"

print("Saving base tables to catalog...\n")

# Save orders_cleaned
orders_cleaned.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.orders_cleaned")
print(f"✓ {catalog}.{schema}.orders_cleaned")

# Save monthly_sales_by_country
monthly_sales_by_country.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.monthly_sales_by_country")
print(f"✓ {catalog}.{schema}.monthly_sales_by_country")

# Save quarterly_sales_by_country
quarterly_sales_by_country.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.quarterly_sales_by_country")
print(f"✓ {catalog}.{schema}.quarterly_sales_by_country")

# Save country_sales
country_sales.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.country_sales")
print(f"✓ {catalog}.{schema}.country_sales")

# Save top_products
top_products.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.top_products")
print(f"✓ {catalog}.{schema}.top_products")

# Save customer_summary
customer_summary.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.customer_summary")
print(f"✓ {catalog}.{schema}.customer_summary")

print("\n🎉 All base tables saved to workspace.ecommerce_analytics!")

Saving base tables to catalog...

✓ workspace.ecommerce_analytics.orders_cleaned
✓ workspace.ecommerce_analytics.monthly_sales_by_country
✓ workspace.ecommerce_analytics.quarterly_sales_by_country
✓ workspace.ecommerce_analytics.country_sales
✓ workspace.ecommerce_analytics.top_products
✓ workspace.ecommerce_analytics.customer_summary

🎉 All base tables saved to workspace.ecommerce_analytics!


In [0]:
# Create and save SQL analysis views as tables

print("Creating analysis tables from SQL queries...\n")

# 1. Monthly Revenue Trends
monthly_revenue_trends = spark.sql("""
SELECT 
  Year,
  Month,
  SUM(total_revenue) as monthly_revenue,
  SUM(order_count) as total_orders,
  ROUND(SUM(total_revenue) / SUM(order_count), 2) as avg_order_value
FROM workspace.ecommerce_analytics.monthly_sales_by_country
GROUP BY Year, Month
ORDER BY Year, Month
""")
monthly_revenue_trends.write.mode("overwrite").saveAsTable("workspace.ecommerce_analytics.analysis_monthly_revenue_trends")
print("✓ workspace.ecommerce_analytics.analysis_monthly_revenue_trends")

# 2. Top Countries Market Share
top_countries_market_share = spark.sql("""
SELECT 
  Country,
  total_revenue,
  total_orders,
  unique_customers,
  avg_order_value,
  ROUND(total_revenue / SUM(total_revenue) OVER () * 100, 2) as market_share_percent
FROM workspace.ecommerce_analytics.country_sales
ORDER BY total_revenue DESC
LIMIT 10
""")
top_countries_market_share.write.mode("overwrite").saveAsTable("workspace.ecommerce_analytics.analysis_top_countries")
print("✓ workspace.ecommerce_analytics.analysis_top_countries")

# 3. Customer Segmentation
customer_segmentation = spark.sql("""
SELECT 
  CASE 
    WHEN total_spent >= 10000 THEN 'High Value'
    WHEN total_spent >= 1000 THEN 'Medium Value'
    ELSE 'Low Value'
  END as customer_segment,
  COUNT(CustomerID) as customer_count,
  ROUND(SUM(total_spent), 2) as segment_revenue,
  ROUND(AVG(total_spent), 2) as avg_customer_value,
  ROUND(AVG(total_orders), 2) as avg_orders_per_customer
FROM workspace.ecommerce_analytics.customer_summary
GROUP BY 
  CASE 
    WHEN total_spent >= 10000 THEN 'High Value'
    WHEN total_spent >= 1000 THEN 'Medium Value'
    ELSE 'Low Value'
  END
ORDER BY segment_revenue DESC
""")
customer_segmentation.write.mode("overwrite").saveAsTable("workspace.ecommerce_analytics.analysis_customer_segments")
print("✓ workspace.ecommerce_analytics.analysis_customer_segments")

# 4. Quarterly Performance
quarterly_performance = spark.sql("""
SELECT 
  Year,
  Quarter,
  SUM(total_revenue) as quarterly_revenue,
  SUM(order_count) as total_orders,
  COUNT(DISTINCT Country) as countries_served,
  ROUND(SUM(total_revenue) / SUM(order_count), 2) as avg_order_value
FROM workspace.ecommerce_analytics.quarterly_sales_by_country
GROUP BY Year, Quarter
ORDER BY Year, Quarter
""")
quarterly_performance.write.mode("overwrite").saveAsTable("workspace.ecommerce_analytics.analysis_quarterly_performance")
print("✓ workspace.ecommerce_analytics.analysis_quarterly_performance")

# 5. Customer Loyalty
customer_loyalty = spark.sql("""
SELECT 
  CASE 
    WHEN total_orders = 1 THEN 'One-time Buyer'
    WHEN total_orders BETWEEN 2 AND 5 THEN 'Occasional (2-5 orders)'
    WHEN total_orders BETWEEN 6 AND 20 THEN 'Regular (6-20 orders)'
    ELSE 'Loyal (20+ orders)'
  END as customer_type,
  COUNT(CustomerID) as customer_count,
  ROUND(SUM(total_spent), 2) as total_revenue,
  ROUND(AVG(total_spent), 2) as avg_lifetime_value,
  ROUND(SUM(total_spent) / SUM(SUM(total_spent)) OVER () * 100, 2) as revenue_contribution_percent
FROM workspace.ecommerce_analytics.customer_summary
GROUP BY 
  CASE 
    WHEN total_orders = 1 THEN 'One-time Buyer'
    WHEN total_orders BETWEEN 2 AND 5 THEN 'Occasional (2-5 orders)'
    WHEN total_orders BETWEEN 6 AND 20 THEN 'Regular (6-20 orders)'
    ELSE 'Loyal (20+ orders)'
  END
ORDER BY total_revenue DESC
""")
customer_loyalty.write.mode("overwrite").saveAsTable("workspace.ecommerce_analytics.analysis_customer_loyalty")
print("✓ workspace.ecommerce_analytics.analysis_customer_loyalty")

# 6. UK vs International
uk_vs_international = spark.sql("""
SELECT 
  Year,
  Month,
  SUM(CASE WHEN Country = 'United Kingdom' THEN total_revenue ELSE 0 END) as uk_revenue,
  SUM(CASE WHEN Country != 'United Kingdom' THEN total_revenue ELSE 0 END) as international_revenue,
  ROUND(SUM(CASE WHEN Country != 'United Kingdom' THEN total_revenue ELSE 0 END) / 
        NULLIF(SUM(total_revenue), 0) * 100, 2) as international_percentage
FROM workspace.ecommerce_analytics.monthly_sales_by_country
GROUP BY Year, Month
ORDER BY Year, Month
""")
uk_vs_international.write.mode("overwrite").saveAsTable("workspace.ecommerce_analytics.analysis_uk_vs_international")
print("✓ workspace.ecommerce_analytics.analysis_uk_vs_international")

print("\n🎉 All analysis tables created successfully!")

Creating analysis tables from SQL queries...

✓ workspace.ecommerce_analytics.analysis_monthly_revenue_trends
✓ workspace.ecommerce_analytics.analysis_top_countries
✓ workspace.ecommerce_analytics.analysis_customer_segments
✓ workspace.ecommerce_analytics.analysis_quarterly_performance
✓ workspace.ecommerce_analytics.analysis_customer_loyalty
✓ workspace.ecommerce_analytics.analysis_uk_vs_international

🎉 All analysis tables created successfully!


In [0]:
# View all tables created in the ecommerce_analytics schema
print("📊 All Tables in workspace.ecommerce_analytics:\n")
print("="*60)

tables = spark.sql("SHOW TABLES IN workspace.ecommerce_analytics").collect()

print("\n📦 BASE TABLES (6):")
for table in tables:
    if not table.tableName.startswith('analysis_'):
        print(f"  • {table.tableName}")

print("\n📈 ANALYSIS TABLES (6):")
for table in tables:
    if table.tableName.startswith('analysis_'):
        print(f"  • {table.tableName}")

print("\n" + "="*60)
print(f"\n✅ Total tables created: {len(tables)}")
print("\n🎯 Ready for Genie Space and Dashboard creation!")

📊 All Tables in workspace.ecommerce_analytics:


📦 BASE TABLES (6):
  • country_sales
  • customer_summary
  • monthly_sales_by_country
  • orders_cleaned
  • quarterly_sales_by_country
  • top_products

📈 ANALYSIS TABLES (6):
  • analysis_customer_loyalty
  • analysis_customer_segments
  • analysis_monthly_revenue_trends
  • analysis_quarterly_performance
  • analysis_top_countries
  • analysis_uk_vs_international


✅ Total tables created: 12

🎯 Ready for Genie Space and Dashboard creation!
